<a href="https://colab.research.google.com/github/Abdo15P/Fairness-Aware-KMeans/blob/main/kmeans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


class K_Means:

    def __init__(self, k=2, tolerance=0.001, max_iter=5000):
        self.k = k
        self.max_iterations = max_iter
        self.tolerance = tolerance
        self.iterations_run = 0
        self.ini_point=None


    def euclidean_distance(self, point1, point2):
        return np.linalg.norm(point1 - point2, axis=0)

    #Predict cluster of a new point
    def predict(self, data):
        distances = [np.linalg.norm(data - self.centroids[idx]) for idx in self.centroids]
        classification = distances.index(min(distances))
        return classification

    #Standard k_means with random initalization
    def k_means_standard(self, data):
        # Pick K random indices from the dataset without replacement
        random_indices = np.random.choice(data.shape[0], 1, replace=False) # choose k random indices from data
        return data[random_indices] # return the chosen subset only

    #K_means++
    def K_Means_pp(self, centroids, data):
        distances_squared_prob = []
        total_sqaure_distances = 0

        # build probability distribution, compute distance from every datapoint to all current centroids,
        # find closest centroid, squared distances become the probabilities.
        for point in data:
            distances_to_centroids = [np.linalg.norm(point - centroids[centroid]) for centroid in range(len(self.centroids))]
            min_distance = min(distances_to_centroids)** 2
            distances_squared_prob.append(min_distance)
            total_sqaure_distances = total_sqaure_distances + min_distance
        distances_squared_prob = np.array(distances_squared_prob) / total_sqaure_distances
        distances_squared_prob = distances_squared_prob / np.sum(distances_squared_prob)
        idx = np.random.choice(data.shape[0], p=distances_squared_prob) #select a random index using computed probabilities
        return data[idx]

    def fit(self, data):
        self.centroids = {}
        self.centroids[0] = data[np.random.randint(len(data))] #select first centorid randomly
        for i in range(1, self.k, 1):
            self.centroids[i] = self.K_Means_pp(self.centroids, data) #select rest of centorids using kmeans or kmeans++
        self.ini_point=self.centroids.copy()

        for i in range(self.max_iterations):
            self.iterations_run = i + 1
            self.classes = {}
            for j in range(self.k):
                self.classes[j] = []

            #find the smallest distance
            for point in data:
                distances = []
                for index in self.centroids:
                    distances.append(self.euclidean_distance(point, self.centroids[index]))
                cluster_index = distances.index(min(distances))
                self.classes[cluster_index].append(point)

            previous = dict(self.centroids) #store previous centorids

            #update the cluster centroid using the mean
            for cluster_index in self.classes:
                self.centroids[cluster_index] = np.average(self.classes[cluster_index], axis=0)

            isOptimal = True

            #continue updating centroids until the difference is below a ceratain tolearnce
            for idx in self.centroids:
                original_centroid = previous[idx]
                current_centroid = self.centroids[idx]
                if np.linalg.norm(current_centroid-original_centroid) / np.linalg.norm(original_centroid) > self.tolerance:
                    isOptimal = False
            if isOptimal:
                break

